In [ ]:
# Colab bootstrap —— 在 Colab 上第一件事就是跑這格。本機開發時它會自動跳過安裝。
#
# --no-deps         Colab 預裝的 torch 是對著它自己的 CUDA 編的，不能讓 pip 重裝
# --force-reinstall 版本號沒變時 pip 會跳過安裝，推了修正之後要靠它抓到新版
#
# 重裝之後舊模組還留在 sys.modules 裡，**要重啟 kernel** 才吃得到新版。
#
# 用 subprocess 而不是 %pip：這樣同一格在本機與 Colab 都能原樣執行。
#
# 三份 notebook 的這一格除了 REQUIRES 之外必須逐字相同，tests/test_notebooks.py 會檢查。
import importlib
import importlib.metadata
import json
import subprocess
import sys

REPO = "git+https://github.com/318amne-Sia/tfn-pytorch.git@main"

# 這份 notebook 會用到的東西，寫成 "模組" 或 "模組:名字"。裝完之後確認它們真的在。
#
# Colab 裝的是 GitHub 上的 main，所以**本機 commit 了卻忘記 push 的程式碼，在這邊
# 並不存在**。少了這個檢查，錯誤會延到後面幾格才炸成一句看不出原因的 ImportError，
# 而且訊息裡的 site-packages 路徑完全指不到真正的原因。
#
# 連名字一起列而不只列模組：模組在、但裡面少一個新加的函式，同樣是沒 push 造成的，
# 而只檢查模組的話這種情況會整個漏掉。
REQUIRES = [
    "tfn.moment_of_inertia",
    "tfn.layers:Y_2",
    "tfn.layers:matrix_from_0_2",
    "tfn.utils:normalized_rmse",
    "tfn.utils:random_rotation_matrix",
]

# 用 try/import 而不是 importlib.util.find_spec("google.colab")：後者在沒有
# google 這個套件的環境（例如本機）會直接拋 ModuleNotFoundError，不是回 None。
try:
    import google.colab  # noqa: F401

    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # 這個檢查刻意放在安裝**之前**：tfn 已經被 import 過的話，裝完也不會生效，
    # 那一趟 git clone 加 wheel build 純屬浪費（在 Colab 上是幾十秒）。
    #
    # 為什麼不靠 __version__ 判斷：它是寫死在原始碼裡的 "0.1.0"，永遠不變，
    # 分不出「剛裝好的新版」與「還留在 sys.modules 的舊版」。
    if "tfn" in sys.modules:
        raise RuntimeError(
            "tfn 已經被 import 過，現在安裝也不會生效。"
            "請先重啟 kernel（執行階段 → 重新啟動工作階段）再跑這格。"
        )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", REPO],
        check=True,
    )
    importlib.invalidate_caches()  # 讓剛寫進 site-packages 的檔案馬上被看見

# 這個 import 刻意放在安裝之後（E402）：Colab 上要先裝好才 import 得到
import tfn  # noqa: E402

# 印出路徑而不只是版本號：本機跑的是工作目錄裡那份，Colab 跑的是 site-packages
print("tfn", tfn.__version__, "來自", tfn.__file__)

if in_colab:
    # 印出實際裝到的 commit，好跟本機的 `git log -1` 對照。pip 從 git URL 安裝時會把
    # 來源記在 dist-info 的 direct_url.json 裡。純診斷用，拿不到就算了。
    try:
        source = json.loads(importlib.metadata.distribution("tfn").read_text("direct_url.json"))
        print("裝到的 commit:", source["vcs_info"]["commit_id"][:10])
    except Exception:
        pass

    # 一次收集全部缺的再報，不要一個一個逼使用者 push、重啟、重跑好幾輪
    missing = []
    for entry in REQUIRES:
        module, _, name = entry.partition(":")
        try:
            imported = importlib.import_module(module)
        except ImportError as error:
            # 缺的若是別的套件（例如 --no-deps 之下的 scipy），那是另一回事，
            # 原樣拋出去。把它說成「沒 push」會把人帶往完全錯的方向。
            culprit = getattr(error, "name", None) or ""
            if not culprit.startswith("tfn"):
                raise
            missing.append(entry)
            continue
        if name and not hasattr(imported, name):
            missing.append(entry)

    if missing:
        raise RuntimeError(
            f"{'、'.join(missing)} 不在剛裝好的套件裡。"
            "GitHub 的 main 還沒有這段程式碼——本機 commit 之後有 push 嗎？\n"
            "push 完的順序是：重啟 kernel → 重跑這一格。"
        )

# 實驗二之二：轉動慣量張量

論文 [Tensor Field Networks](https://arxiv.org/abs/1802.08219) §5.2 的後半，
也是 README 表格裡列的實驗二。

給一堆隨機點和它們的質量，指定其中一個當中心，問**繞著那個中心轉有多難**。
繞不同軸的難度不一樣，所以答案不是一個數字，是一個 3×3 對稱矩陣：

$$I_{ij} = \sum_p m_p \left( |\vec r_p|^2 \delta_{ij} - (\vec r_p)_i (\vec r_p)_j \right)$$

網路型別是 $0 \to 0 \oplus 2$：餵進去一個數字（質量），要吐出**兩樣東西**
再拼成矩陣。下一節解釋為什麼是兩樣。

In [ ]:
import numpy as np
import torch

from tfn import moment_of_inertia as moi
from tfn.layers import Y_2, matrix_from_0_2
from tfn.utils import normalized_rmse, random_rotation_matrix

SEED = 0

points, masses = moi.random_points_and_masses(SEED)
print(f"{len(points)} 個點，中心是第 {moi.CENTRE_INDEX} 個")
for i, (p, m) in enumerate(zip(points.tolist(), masses.tolist(), strict=True)):
    mark = "  ← 中心，質量歸零" if i == moi.CENTRE_INDEX else ""
    print(f"  ({p[0]:+.2f}, {p[1]:+.2f}, {p[2]:+.2f})   質量 {m:.2f}{mark}")

## 為什麼輸出是「0 和 2」兩樣東西

一個 3×3 矩陣有 9 個數字，但這 9 個在旋轉之下**不是一體的**。照 $M \to R M R^\top$
實際去算，它們會分成三堆，每堆只跟自己人混：

| | 個數 | 是什麼 |
| --- | --- | --- |
| L=0 | 1 | trace，整體大小 |
| L=1 | 3 | 反對稱部分 |
| L=2 | 5 | 無 trace 的對稱部分，形狀 |

轉動慣量張量是**對稱的**，所以中間那 3 個恆為 0，只剩 $1 + 5 = 6$——正好是
對稱矩陣的獨立數字個數。

`matrix_from_0_2` 就是把這 1 + 5 換寫成 3×3。它**沒有參數**，全是固定常數，
而且和 `Y_2` 合起來有一個封閉形式——下面這格就是在驗這件事。

（為什麼中間不能放一個可學的全連接層？因為任意權重會把「旋轉時不動的那 1 個」
和「旋轉時會轉的那 5 個」加在一起，加完就哪條規則都不遵守了。）

In [ ]:
direction = torch.tensor([0.3, -0.5, 0.8])
unit = direction / direction.norm()

built = matrix_from_0_2(torch.tensor(0.0), Y_2(direction))
closed_form = torch.outer(unit, unit) - torch.eye(3) / 3

print("matrix_from_0_2(0, Y_2(r))  與  r̂r̂ᵀ − I/3")
print(f"  最大逐元素差：{(built - closed_form).abs().max():.2e}")
print()

# 第二條恆等式：那 5 個數字對 trace 完全沒有影響力
scalar = torch.tensor(2.5)
noise = torch.randn(5)
print(f"隨便丟 5 個 L=2 的數字，trace = {matrix_from_0_2(scalar, noise).trace():.6f}")
print(f"3 × 純量                      = {3 * scalar:.6f}")
print("→ L=2 那半動不了 trace，這正是 1 + 5 拆法成立的前提。")

## 等變性是結構帶來的，不是訓練來的

**還沒訓練**的網路，把座標轉一個角度，輸出的矩陣就照 $R M R^\top$ 跟著轉。
解析解自己也必須如此——否則就是拿一組會破壞等變性的標準答案去教網路。

In [ ]:
torch.manual_seed(SEED)
untrained = moi.MomentOfInertiaModel()

pts, ms = moi.random_points_and_masses(1)
rotation = random_rotation_matrix(7)

with torch.no_grad():
    model_gap = (
        (untrained(pts @ rotation.T, ms) - rotation @ untrained(pts, ms) @ rotation.T).abs().max()
    )

truth_gap = (
    (
        moi.moment_of_inertia(pts @ rotation.T, ms)
        - rotation @ moi.moment_of_inertia(pts, ms) @ rotation.T
    )
    .abs()
    .max()
)

print(f"未訓練網路   最大偏差：{model_gap:.2e}")
print(f"解析解       最大偏差：{truth_gap:.2e}")

out = untrained(pts, ms)
print(f"\n輸出對稱嗎（結構保證，不是學來的）：{torch.equal(out, out.transpose(-2, -1))}")

## 資料，以及論文圖上標錯的那條線

每步現場亂生 15 個點，座標每軸 ±0.5。**中心點的質量設成 0**——因為 L=0 濾波器
沒有「自己對自己」的遮罩，不歸零的話中心點自己的質量會經由 $R_0(0)$ 漏進輸出。

上游 notebook 的圖把距離上限標在 $\sqrt{3 \cdot 0.5^2} = 0.866$。那是**點到原點**
的最大距離——但相關的是「中心點到其他點」的距離，而中心點自己也是隨機的。
下面這格量給你看。

In [ ]:
rng = np.random.default_rng(0)
separations = []
for _ in range(5000):
    pts, _ = moi.random_points_and_masses(rng)
    separations.append(torch.linalg.vector_norm(pts[1:] - pts[moi.CENTRE_INDEX], dim=-1))
separations = torch.cat(separations)

p5, p95 = np.percentile(separations.numpy(), [5, 95])
actual_max = separations.max().item()
print(f"中心點到其他點的距離：中位數 {separations.median():.2f}，最大 {separations.max():.2f}")
print(f"p5 = {p5:.2f}，p95 = {p95:.2f}   ← 比對區間取這一段，涵蓋 90% 的資料")
print()
print(f"落在論文標的 0.866 之外的比例：{(separations > 0.866).float().mean():.1%}")
print("  0.866 = sqrt(3)·0.5 是「點到原點」的最大距離，不是點對之間的。")
print(f"  真正的最大值是 {actual_max:.2f}。")

## 訓練

單層、兩條並排的路徑（L=0 與 L=2）、各一個通道。沒有 self-interaction、沒有
非線性、最後也沒有全連接層。Adam `lr = 1e-4`，10001 步。

loss 是差的平方和除以 2，而且**只取中心點那一個 3×3**——網路對 15 個點都會算
出矩陣，其餘 14 個丟掉。轉動慣量是「繞著某個中心」才有定義的。

In [ ]:
torch.manual_seed(SEED)
model = moi.MomentOfInertiaModel()
history = moi.train(model, steps=moi.TRAIN_STEPS, rng=SEED)

for step in range(0, moi.TRAIN_STEPS, 2000):
    window = history[max(0, step - 199) : step + 1]
    print(f"step {step:6d}   最近 {len(window):3d} 步平均 loss {np.mean(window):9.5f}")

print()
print(f"整個網路的參數量：{sum(p.numel() for p in model.parameters())}")
print("（兩條徑向函數，各 961 個參數。全部的學習都發生在這裡。）")

## 驗收：兩條徑向函數

這就是論文 §5.2 的成果。沒有人告訴網路 $\frac{2}{3}r^2$ 和 $-r^2$——它只看過
點的座標、質量、和轉動慣量張量。

這兩個係數的來歷是把物理公式拆成 trace 與非 trace 兩半：trace 那份是
$\frac{2}{3}\sum m r^2$，剩下那份是 $-\sum m r^2 (\hat r \hat r^\top - I/3)$。
網路裡沒有別的自由度能吸收它們，所以曲線的尺度是絕對釘死的。

In [ ]:
# 圖上的文字一律用英文：Colab 的 matplotlib 預設字型沒有中文字符。
import matplotlib.pyplot as plt

PAPER_MAX = 0.866  # 上游標的上限，見上面那格

distances = torch.linspace(0.0, 1.6, 400)
learned_0, learned_2 = model.radial_curves(distances)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.axvspan(moi.FIT_LOW, moi.FIT_HIGH, color="#888", alpha=0.10)

ax.plot(
    distances,
    moi.analytic_radial_0(distances),
    "--",
    lw=2.5,
    color="#c0504d",
    label=r"analytic  $\frac{2}{3}r^2$   (L=0)",
)
ax.plot(distances, learned_0, lw=2, color="#c0504d", alpha=0.85, label="learned  (L=0)")
ax.plot(
    distances,
    moi.analytic_radial_2(distances),
    "--",
    lw=2.5,
    color="#2f7d7b",
    label=r"analytic  $-r^2$   (L=2)",
)
ax.plot(distances, learned_2, lw=2, color="#2f7d7b", alpha=0.85, label="learned  (L=2)")

ax.axvline(moi.FIT_LOW, color="#555", ls=":", lw=1.3)
ax.axvline(moi.FIT_HIGH, color="#555", ls=":", lw=1.3)
ax.axvline(PAPER_MAX, color="#b07d2b", ls="-.", lw=1.6)
ax.annotate(
    f'paper\'s "max distance": {PAPER_MAX}\nactual maximum: {actual_max:.2f}',
    xy=(PAPER_MAX, -1.55),
    xytext=(PAPER_MAX + 0.04, -1.55),
    fontsize=8.5,
    color="#8a5f18",
    ha="left",
    va="center",
)
ax.text(
    (moi.FIT_LOW + moi.FIT_HIGH) / 2,
    -2.6,
    "comparison range  0.26 - 1.07   (measured p5 - p95)",
    fontsize=8.5,
    color="#444",
    ha="center",
    va="bottom",
)

ax.axhline(0, color="#999", lw=0.8)
ax.set_xlim(0, 1.6)
ax.set_ylim(-2.8, 1.8)
ax.set_xlabel("distance from the centre point  r")
ax.set_ylabel("radial function output")
ax.set_title("Moment of inertia: both radial functions recover their formulas")
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
ax.grid(alpha=0.15)
plt.show()

## 把「疊得上」變成數字

跟重力那半同一套指標：normalized RMSE。整條曲線符號反轉會得到 2.0，
所以任何合理的門檻都擋得住「CG 係數弄反」那一類錯誤。

In [ ]:
fit = torch.linspace(moi.FIT_LOW, moi.FIT_HIGH, 50)
fit_0, fit_2 = model.radial_curves(fit)

torch.manual_seed(SEED)
fresh = moi.MomentOfInertiaModel()
fresh_0, fresh_2 = fresh.radial_curves(fit)

rows = [
    (
        "L=0  (2/3·r²)",
        normalized_rmse(fit_0, moi.analytic_radial_0(fit)).item(),
        normalized_rmse(fresh_0, moi.analytic_radial_0(fit)).item(),
        moi.RADIAL_L0_NRMSE_CEILING,
    ),
    (
        "L=2  (−r²)",
        normalized_rmse(fit_2, moi.analytic_radial_2(fit)).item(),
        normalized_rmse(fresh_2, moi.analytic_radial_2(fit)).item(),
        moi.RADIAL_L2_NRMSE_CEILING,
    ),
]
print(f"{'':16s}{'訓練後':>10s}{'未訓練':>10s}{'門檻':>10s}")
for name, trained, untrained_error, ceiling in rows:
    print(f"{name:16s}{trained:10.4f}{untrained_error:10.4f}{ceiling:10.2f}")

print("\n（符號整條反轉會是 2.0000）")
print(
    f"\nvalidation loss  {moi.validation_loss(model, samples=moi.VALIDATION_SAMPLES, rng=1):.5f}"
    f"   門檻 {moi.VALIDATION_LOSS_CEILING}"
)